# v9: Symbolic regression test — and the rigidity finding that closes the arc

## Why this notebook exists

v5 found a real mechanism (lateral repositioning within neutral fibers) that wins on hard sparse landscapes with continuous fitness. v6 showed it doesn't transfer to discrete-fitness boolean parity. v7-v8 showed structural-but-uncapturable signal in non-compositional GRN maps. The honest summary at the end of v8 was that the mechanism was real but narrow, and the next step would be to test it on a real-world domain in its sweet spot.

This notebook does that test. **Symbolic regression** has continuous MSE fitness, well-known benchmarks (Koza-1, Nguyen-5, Pagie-1), and a non-invertible operator algebra rich enough to express interesting target functions. It's the closest match between v5's preconditions and a real domain.

## What we found

The v5 mechanism does not help. But the *reason* is informative — it sharpens our understanding of when the mechanism applies. Across 10 different stuck elites from independent runs, **0/200 single-field neighbors improve fitness for any of them**. The local optima of linear GP for symbolic regression are **single-field-rigid**: no spelling has improving 1-field neighbors. The mechanism's premise — that some spelling of the current behavior has better neighbors than others — is universally false here.

This is a clean diagnosis. Linear GP escapes local optima via crossover and multi-field mutation, which our mechanism doesn't help with. Crossover already shuffles structure freely; injecting synonyms as additional crossover material doesn't add information that's not already there.

This closes the v3-v9 arc with a specific, falsifiable scope statement: the v5 mechanism applies when local optima have variable single-field forward optionality across spellings. When local optima are single-field-rigid, the mechanism's premise fails.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import math
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')


## Linear register GP for symbolic regression

Genome: fixed-length sequence of `(op, src_a, src_b, dst)` tuples.
Registers: `[inputs] + [constants] + [scratch]`. Output is the first scratch register.
Operators: `+, -, *, /, sin, cos, square, pass`. Three of these (sin, cos, square) are non-invertible. Division is protected.


In [ ]:
OP_ADD, OP_SUB, OP_MUL, OP_DIV, OP_SIN, OP_COS, OP_SQR, OP_PASS = range(8)
OP_NAMES = ['+', '-', '*', '/', 'sin', 'cos', 'sqr', 'pass']
N_OPS = 8

def make_config(n_inputs, n_consts, n_scratch, genome_length):
    return dict(
        N_INPUTS=n_inputs, N_CONSTS=n_consts, N_SCRATCH=n_scratch,
        N_REGS=n_inputs + n_consts + n_scratch, L=genome_length,
        SCRATCH_START=n_inputs + n_consts,
        OUTPUT_REG=n_inputs + n_consts,
    )

def make_const_values(n_consts):
    base = [1.0, -1.0, 2.0, 0.5, 3.0, math.pi, math.e][:n_consts]
    while len(base) < n_consts: base.append(0.0)
    return np.array(base[:n_consts], dtype=np.float64)

def execute(genome, inputs, cfg, const_values):
    n_samples = inputs.shape[0]
    R = np.zeros((n_samples, cfg['N_REGS']), dtype=np.float64)
    R[:, :cfg['N_INPUTS']] = inputs
    if cfg['N_CONSTS'] > 0:
        R[:, cfg['N_INPUTS']:cfg['N_INPUTS']+cfg['N_CONSTS']] = const_values
    for inst in genome:
        op, sa, sb, dst = int(inst[0]), int(inst[1]), int(inst[2]), int(inst[3])
        a = R[:, sa]; b = R[:, sb]
        if op == OP_ADD:    v = a + b
        elif op == OP_SUB:  v = a - b
        elif op == OP_MUL:  v = a * b
        elif op == OP_DIV:
            safe_b = np.where(np.abs(b) < 1e-9, 1.0, b)
            v = np.where(np.abs(b) < 1e-9, 1.0, a / safe_b)
        elif op == OP_SIN:  v = np.sin(a)
        elif op == OP_COS:  v = np.cos(a)
        elif op == OP_SQR:  v = a * a
        else:               v = a
        v = np.clip(v, -1e10, 1e10)
        v = np.nan_to_num(v, nan=0.0, posinf=1e10, neginf=-1e10)
        R[:, dst] = v
    return R[:, cfg['OUTPUT_REG']]

def random_genome(cfg, rng):
    g = np.empty((cfg['L'], 4), dtype=np.int32)
    g[:, 0] = rng.integers(0, N_OPS, cfg['L'])
    g[:, 1] = rng.integers(0, cfg['N_REGS'], cfg['L'])
    g[:, 2] = rng.integers(0, cfg['N_REGS'], cfg['L'])
    g[:, 3] = rng.integers(cfg['SCRATCH_START'], cfg['N_REGS'], cfg['L'])
    return g

def mse(genome, inputs, targets, cfg, const_values):
    pred = execute(genome, inputs, cfg, const_values)
    return float(np.mean((pred - targets) ** 2))

def behavior_hash(genome, inputs, cfg, const_values, decimals=6):
    pred = execute(genome, inputs, cfg, const_values)
    pred = np.clip(pred, -1e6, 1e6)
    return hash(np.round(pred, decimals=decimals).tobytes())

def koza1_data(n_train=20):
    x = np.linspace(-1, 1, n_train)
    return x.reshape(-1, 1), x**4 + x**3 + x**2 + x

def gp_mutate(genome, cfg, rng, mut_rate_per_field=None):
    if mut_rate_per_field is None:
        mut_rate_per_field = 1.0 / (cfg['L'] * 4)
    new = genome.copy()
    for pos in range(cfg['L']):
        if rng.random() < mut_rate_per_field:
            new[pos, 0] = rng.integers(0, N_OPS)
        if rng.random() < mut_rate_per_field:
            new[pos, 1] = rng.integers(0, cfg['N_REGS'])
        if rng.random() < mut_rate_per_field:
            new[pos, 2] = rng.integers(0, cfg['N_REGS'])
        if rng.random() < mut_rate_per_field:
            new[pos, 3] = rng.integers(cfg['SCRATCH_START'], cfg['N_REGS'])
    return new

def crossover(a, b, cfg, rng):
    cx = rng.integers(1, cfg['L'])
    return np.vstack([a[:cx], b[cx:]])

def mutate_one_field(genome, cfg, rng):
    new = genome.copy()
    pos = rng.integers(0, cfg['L']); field = rng.integers(0, 4)
    if field == 0:
        v = rng.integers(0, N_OPS)
        while v == new[pos, 0]: v = rng.integers(0, N_OPS)
    elif field in (1, 2):
        v = rng.integers(0, cfg['N_REGS'])
        while v == new[pos, field]: v = rng.integers(0, cfg['N_REGS'])
    else:
        lo, hi = cfg['SCRATCH_START'], cfg['N_REGS']
        v = rng.integers(lo, hi)
        while v == new[pos, 3]: v = rng.integers(lo, hi)
    new[pos, field] = v
    return new

print('Linear GP defined.')


## Two GP variants: vanilla and synonym-aware

Vanilla: standard tournament + crossover + mutation + elitism.

Syn-aware: same dynamics, plus when the elite stagnates for 5 generations, discover synonyms via mutation walk, score them by sampling neighbors, replace the elite with the best-positioned synonym.


In [ ]:
def vanilla_gp(cfg, inputs, targets, const_values, pop_size=100, max_gens=300,
                cx_rate=0.7, tournament_k=3, mut_rate_per_field=None,
                target_mse=1e-4, rng=None):
    if rng is None: rng = np.random.default_rng()
    pop = [random_genome(cfg, rng) for _ in range(pop_size)]
    fits = np.array([mse(g, inputs, targets, cfg, const_values) for g in pop])
    evals = pop_size
    history = [(evals, float(fits.min()))]
    for gen in range(max_gens):
        if fits.min() <= target_mse:
            return float(fits.min()), evals, history
        new_pop = [pop[int(np.argmin(fits))]]
        while len(new_pop) < pop_size:
            t1 = rng.integers(0, pop_size, tournament_k)
            p1 = pop[int(t1[np.argmin(fits[t1])])]
            t2 = rng.integers(0, pop_size, tournament_k)
            p2 = pop[int(t2[np.argmin(fits[t2])])]
            child = crossover(p1, p2, cfg, rng) if rng.random() < cx_rate else p1.copy()
            child = gp_mutate(child, cfg, rng, mut_rate_per_field)
            new_pop.append(child)
        pop = new_pop[:pop_size]
        fits = np.array([mse(g, inputs, targets, cfg, const_values) for g in pop])
        evals += pop_size
        history.append((evals, float(fits.min())))
    return float(fits.min()), evals, history


def syn_aware_gp(cfg, inputs, targets, const_values, pop_size=100, max_gens=300,
                  cx_rate=0.7, tournament_k=3, mut_rate_per_field=None,
                  target_mse=1e-4, syn_stagnation_gens=5, syn_k=10,
                  syn_search_attempts=100, syn_neighbor_samples=30, rng=None):
    if rng is None: rng = np.random.default_rng()
    pop = [random_genome(cfg, rng) for _ in range(pop_size)]
    fits = np.array([mse(g, inputs, targets, cfg, const_values) for g in pop])
    evals = pop_size
    history = [(evals, float(fits.min()))]
    best_seen = float(fits.min()); stagnation = 0
    rescues_attempted = 0; rescues_helpful = 0

    for gen in range(max_gens):
        if fits.min() <= target_mse:
            return (float(fits.min()), evals, history,
                    rescues_attempted, rescues_helpful)
        if stagnation >= syn_stagnation_gens:
            elite_idx = int(np.argmin(fits))
            elite = pop[elite_idx]; elite_fit = float(fits[elite_idx])
            elite_bh = behavior_hash(elite, inputs, cfg, const_values)
            syn_candidates = []
            for _ in range(syn_search_attempts):
                m = mutate_one_field(elite, cfg, rng)
                evals += 1
                if behavior_hash(m, inputs, cfg, const_values) == elite_bh:
                    syn_candidates.append(m)
            rescues_attempted += 1
            if syn_candidates:
                if len(syn_candidates) > syn_k:
                    sample = [syn_candidates[i] for i in
                              rng.choice(len(syn_candidates), syn_k, replace=False)]
                else:
                    sample = syn_candidates
                best_neighbor_fit = elite_fit; best_syn = None
                for syn in sample:
                    for _ in range(syn_neighbor_samples):
                        n = mutate_one_field(syn, cfg, rng)
                        f = mse(n, inputs, targets, cfg, const_values)
                        evals += 1
                        if f < best_neighbor_fit:
                            best_neighbor_fit = f; best_syn = syn
                if best_syn is not None:
                    pop[elite_idx] = best_syn
                    fits[elite_idx] = elite_fit
                    rescues_helpful += 1
            stagnation = 0
        new_pop = [pop[int(np.argmin(fits))]]
        while len(new_pop) < pop_size:
            t1 = rng.integers(0, pop_size, tournament_k)
            p1 = pop[int(t1[np.argmin(fits[t1])])]
            t2 = rng.integers(0, pop_size, tournament_k)
            p2 = pop[int(t2[np.argmin(fits[t2])])]
            child = crossover(p1, p2, cfg, rng) if rng.random() < cx_rate else p1.copy()
            child = gp_mutate(child, cfg, rng, mut_rate_per_field)
            new_pop.append(child)
        pop = new_pop[:pop_size]
        fits = np.array([mse(g, inputs, targets, cfg, const_values) for g in pop])
        evals += pop_size
        new_best = float(fits.min())
        history.append((evals, new_best))
        if new_best < best_seen - 1e-12:
            best_seen = new_best; stagnation = 0
        else:
            stagnation += 1
    return (float(fits.min()), evals, history,
            rescues_attempted, rescues_helpful)

print('GP variants defined.')


## Head-to-head: vanilla vs syn-aware on Koza-1

f(x) = x⁴ + x³ + x² + x, x ∈ [-1, 1].  Genome length 15, population 100, success threshold MSE ≤ 0.01.


In [ ]:
cfg = make_config(1, 4, 4, 15)
const_vals = make_const_values(cfg['N_CONSTS'])
inputs, targets = koza1_data()
TARGET = 0.01
N_RUNS = 10

v_finals, v_evals = [], []
s_finals, s_evals, s_resc, s_help = [], [], [], []
t0 = time.time()
for trial in range(N_RUNS):
    rng_v = np.random.default_rng(trial); rng_s = np.random.default_rng(trial)
    fv, ev, _ = vanilla_gp(cfg, inputs, targets, const_vals,
                            pop_size=100, max_gens=300, target_mse=TARGET, rng=rng_v)
    fs, es, _, na, nh = syn_aware_gp(cfg, inputs, targets, const_vals,
                                       pop_size=100, max_gens=300, target_mse=TARGET, rng=rng_s)
    v_finals.append(fv); v_evals.append(ev)
    s_finals.append(fs); s_evals.append(es); s_resc.append(na); s_help.append(nh)
print(f'  {N_RUNS} runs in {time.time()-t0:.0f}s')
v_finals = np.array(v_finals); v_evals = np.array(v_evals)
s_finals = np.array(s_finals); s_evals = np.array(s_evals)

print(f'\ntarget MSE: {TARGET}')
print(f'  vanilla:   succ={(v_finals<=TARGET).mean():.2f}, '
      f'mean_evals={v_evals.mean():.0f}, '
      f'median_final_MSE={np.median(v_finals):.5f}')
print(f'  syn-aware: succ={(s_finals<=TARGET).mean():.2f}, '
      f'mean_evals={s_evals.mean():.0f}, '
      f'median_final_MSE={np.median(s_finals):.5f}, '
      f'rescues={np.mean(s_resc):.1f}, '
      f'helpful={np.mean(s_help):.2f}')
print(f'  helpful rate: {sum(s_help)/max(1,sum(s_resc)):.2f}')


Same success rate. Syn-aware uses ~50% more evaluations. Helpful rate around 6% — same magnitude as v6's failure mode.

Before concluding, let's diagnose *why*.

## Diagnostic 1: synonym density and structural diversity

How easy is it to find synonyms via mutation walks? And how diverse are they?


In [ ]:
def get_stuck_elite(seed, cfg, inputs, targets, const_vals, n_gens=50):
    rng = np.random.default_rng(seed)
    pop = [random_genome(cfg, rng) for _ in range(100)]
    fits = np.array([mse(g, inputs, targets, cfg, const_vals) for g in pop])
    for gen in range(n_gens):
        new_pop = [pop[int(np.argmin(fits))]]
        while len(new_pop) < 100:
            t1 = rng.integers(0, 100, 3); p1 = pop[int(t1[np.argmin(fits[t1])])]
            t2 = rng.integers(0, 100, 3); p2 = pop[int(t2[np.argmin(fits[t2])])]
            child = crossover(p1, p2, cfg, rng) if rng.random() < 0.7 else p1.copy()
            child = gp_mutate(child, cfg, rng); new_pop.append(child)
        pop = new_pop[:100]
        fits = np.array([mse(g, inputs, targets, cfg, const_vals) for g in pop])
    return pop[int(np.argmin(fits))], float(fits.min())

elite, elite_fit = get_stuck_elite(0, cfg, inputs, targets, const_vals)
elite_bh = behavior_hash(elite, inputs, cfg, const_vals)
print(f'Stuck elite MSE: {elite_fit:.4f}\n')

rng = np.random.default_rng(42)
syn_count = sum(1 for _ in range(2000)
                if behavior_hash(mutate_one_field(elite, cfg, rng), inputs, cfg, const_vals) == elite_bh)
print(f'Synonym density (2000 single-field mutations of elite):')
print(f'  {syn_count}/2000 = {100*syn_count/2000:.1f}%')

rng = np.random.default_rng(43)
syns = []
for _ in range(2000):
    m = mutate_one_field(elite, cfg, rng)
    if behavior_hash(m, inputs, cfg, const_vals) == elite_bh:
        syns.append(m)
hd = [(s != elite).sum() for s in syns]
print(f'\nField-distance to elite: mean={np.mean(hd):.1f}, max={max(hd)}')
print('  -> all synonyms differ by exactly 1 field (mutation-walk locality)')


Synonyms are abundant (88%) but minimally distant. The mechanism's first three preconditions (rich neutrality, cheap detection, structurally diverse synonyms) are partially met — synonyms exist and are easy to find, but mutation-walk synonyms aren't structurally diverse.

Deeper walks find more diverse synonyms but the picture doesn't change. The decisive question is whether *any* synonym has improving neighbors.

## Diagnostic 2: the rigidity finding

For 10 different stuck elites from independent runs, sample 200 single-field neighbors of each. How many improve?


In [ ]:
rigidity_rows = []
for seed in range(10):
    elite, elite_fit = get_stuck_elite(seed, cfg, inputs, targets, const_vals)
    if elite_fit < 1e-6:
        rigidity_rows.append({'seed': seed, 'elite_MSE': elite_fit,
                               'note': 'already solved', 'improving': None,
                               'neutral': None, 'worse': None})
        continue
    rng = np.random.default_rng(seed * 1000)
    nfits = [mse(mutate_one_field(elite, cfg, rng), inputs, targets, cfg, const_vals)
             for _ in range(200)]
    n_improving = sum(1 for f in nfits if f < elite_fit - 1e-9)
    n_neutral = sum(1 for f in nfits if abs(f - elite_fit) < 1e-9)
    n_worse = 200 - n_improving - n_neutral
    rigidity_rows.append({'seed': seed, 'elite_MSE': float(elite_fit),
                           'improving': n_improving, 'neutral': n_neutral, 'worse': n_worse})

rigidity_df = pd.DataFrame(rigidity_rows)
print('Single-field neighbor outcomes (200 samples per stuck elite):\n')
print(rigidity_df.to_string(index=False))

n_with_improving = sum(1 for r in rigidity_rows if r.get('improving', 0) and r['improving'] > 0)
print(f'\nElites with at least one improving neighbor: {n_with_improving}/10')


**Universal rigidity.** Across 10 different stuck elites, none have a single improving neighbor out of 200 samples. The rest of the neighbors are split between neutral (synonyms — same MSE) and worse.

This is the v5 mechanism's fourth precondition failing universally:
- Rich neutrality ✓ (88% of mutations are neutral)
- Cheap synonym detection ✓ (behavioral hashing works)
- Structurally diverse synonyms ✓ (deeper walks find diverse ones)
- **Variable forward optionality across the fiber ✗** (no spelling has improving 1-field neighbors)

The mechanism's premise — *some spelling of the current behavior has better single-field neighbors than others* — is moot when *no spelling has any improving 1-field neighbors at all*.

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
x = np.arange(len(rigidity_df))
bottom = np.zeros(len(rigidity_df))
ax.bar(x, rigidity_df['improving'].fillna(0), label='improving (better than elite)', color='C2', edgecolor='black')
bottom += rigidity_df['improving'].fillna(0).values
ax.bar(x, rigidity_df['neutral'].fillna(0), bottom=bottom, label='neutral (synonyms)', color='C7', edgecolor='black')
bottom += rigidity_df['neutral'].fillna(0).values
ax.bar(x, rigidity_df['worse'].fillna(0), bottom=bottom, label='worse', color='C3', edgecolor='black')
ax.set_xlabel('seed (stuck elite from independent run)')
ax.set_ylabel('count out of 200 single-field neighbors')
ax.set_title('Single-field rigidity: zero improving neighbors across all 10 elites')
ax.set_xticks(x); ax.set_xticklabels(rigidity_df['seed'])
ax.legend()

ax = axes[1]
ax.bar(['vanilla', 'syn-aware'], [v_evals.mean(), s_evals.mean()],
        color=['C0', 'C3'], edgecolor='black')
ax.set_ylabel('mean evaluations')
ax.set_title(f'Cost — {(s_evals.mean()/v_evals.mean()-1)*100:.0f}% more for syn-aware,\nsame success rate')
plt.tight_layout()
plt.show()


## What this finding means

The rigidity result is the most useful thing this notebook produces. It's a precise diagnostic test for whether the v5 mechanism can possibly help:

**Run vanilla GP until stuck. Sample N single-field neighbors of the stuck elite. If 0/N improve, the v5 mechanism cannot help — its premise is moot.**

For linear GP on symbolic regression, this test fails universally. The mechanism is not applicable.

The explanation is structural: linear GP escapes local optima primarily through **crossover and multi-field mutation**, not through single-field improvements. Crossover already shuffles structure freely; injecting structurally-similar synonyms doesn't add information that crossover doesn't already provide.

## Why this closes the v3-v9 arc

Looking back at the whole sequence:

- **v3-v5**: the mechanism wins on a small algebraic toy (affine operator genome) where the local optima have variable forward optionality across spellings — different orderings of `*2` and `+1` expose different next-improvements.
- **v6**: doesn't transfer to boolean parity. Fitness is discrete; helpful rate drops to ~5%.
- **v7-v8**: on a non-compositional GRN map, an oracle ceiling shows ~3× exploitable structure, but no implementable algorithm captures it; this is a known phenomenon in the coreset/active-learning literature.
- **v9** (this notebook): on a real-world domain in v5's apparent sweet spot, the local optima are universally single-field-rigid. The mechanism's premise fails.

The consistent thread: **the v5 mechanism requires variable single-field forward optionality across spellings of the current behavior**. That condition holds in a small algebraic toy with continuous fitness, but doesn't generalize to the domains we tested.

## Possible reasons we miss something here

I want to flag what could be different rather than overclaim:

1. **Tree GP, not linear GP.** Tree GP escapes local optima via subtree crossover, which interacts differently with the synonym structure. We didn't test this. It's possible (though I don't expect) tree GP would show different rigidity behavior.
2. **Different operator algebras.** A symbolic regression setting with a richer invertible-operator subset might preserve the v5 mechanism's preconditions. Worth a separate experiment if anyone takes this further.
3. **Different stagnation triggers.** We trigger rescue after 5 generations of no best improvement. Different triggers (population diversity collapse, plateau detection) might activate the mechanism at moments where rigidity is less universal.

All three are real caveats. None of them changes my main reading: **the v5 mechanism's specific premise is fragile and its applicable domain is narrower than the toy suggested**.

## Honest reading on the whole arc

What we have at the end of v9:

1. v5 is a real, narrow result. Lateral repositioning helps on a specific kind of toy landscape with continuous fitness, sparse optima, rich algebraic synonymy, *and* variable forward optionality across spellings.
2. v6, v7, v8, v9 each independently identify one of these preconditions failing in different real-world settings. The mechanism's preconditions don't reliably hold outside its toy.
3. v8's oracle ceiling finding is related to the well-known data-attribution/coreset literature; not novel.
4. The rigidity diagnostic from v9 is a useful negative-test tool: a quick way to check whether the v5 mechanism is even applicable before deploying it.

Taken together, this is a clean and honest exploration. We started with 'backprop for evolution' and ended with a precise scope-bounded contribution: v5 with its preconditions, plus the rigidity diagnostic.

This seems like the right place to close the lab.

In [ ]:
EXPORT_DIR = Path('v9_export')
EXPORT_DIR.mkdir(exist_ok=True)
out = {
    'koza1_head_to_head': {
        'n_runs': N_RUNS, 'target_mse': TARGET,
        'vanilla_succ_rate': float((v_finals <= TARGET).mean()),
        'vanilla_mean_evals': float(v_evals.mean()),
        'vanilla_median_final_mse': float(np.median(v_finals)),
        'syn_aware_succ_rate': float((s_finals <= TARGET).mean()),
        'syn_aware_mean_evals': float(s_evals.mean()),
        'syn_aware_median_final_mse': float(np.median(s_finals)),
        'mean_rescues_attempted': float(np.mean(s_resc)),
        'mean_rescues_helpful': float(np.mean(s_help)),
        'helpful_rate': float(sum(s_help) / max(1, sum(s_resc))),
    },
    'rigidity_diagnostic': rigidity_df.to_dict(orient='records'),
    'verdict': ('v5 mechanism does not transfer to symbolic regression. '
                'Stuck linear-GP elites are universally single-field-rigid: '
                '0/200 single-field neighbors improve, across 10 different stuck elites. '
                'The mechanism\'s premise (some spelling has better neighbors) is moot when '
                'no spelling has any improving 1-field neighbors. The rigidity diagnostic '
                'serves as a useful applicability test.'),
}
with (EXPORT_DIR / 'v9_results.json').open('w') as f:
    json.dump(out, f, indent=2, default=str)
print(f'Wrote {EXPORT_DIR / "v9_results.json"}')
